# Data Exploration and Feature Engineering

Explore PFR training data and define **inputs** (`df_features`) and **outputs** (`df_target`) for ML. Run cells in order.


## 1. Setup

Import libraries and set plot style. **Run this cell first.**


In [1]:
import os
import json
import pickle
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
print("Libraries imported successfully.")
plt.style.use('seaborn-v0_8-darkgrid')

Libraries imported successfully.


**Config.** Set data/metadata filenames and `BASE_DIR_ALL`; toggles `IF_LOAD_DATA`, `PRINT_ON_SCREEN`, `IF_SAVE_FIGURES`.

In [2]:
PRINT_ON_SCREEN = True
IF_LOAD_DATA = True
BASE_DIR_ALL = '../data/training'
CORE_DATA_FILE_NAME = "training_data_complete_20260116_224238.pkl" # do not overwrite 
META_DATA_FILE_NAME = "metadata_20260116_224238.json"

IF_EXPORT_FEATURES_TARGETS = True
EXPORT_DIR = "../data/processed/"
EXPORT_FEATURES_TARGETS_FILE_NAME = f"features_targets_{CORE_DATA_FILE_NAME.replace('.pkl', '')}.pkl"

print(50*"=")
print('USER IO FLAGS:')
print(f'LOAD_DATA = {IF_LOAD_DATA}')
print(f'CORE DATA FILE NAME = {CORE_DATA_FILE_NAME}')
print(f'META DATA FILE NAME = {META_DATA_FILE_NAME}')
print(f'BASE DIR ALL = {BASE_DIR_ALL}')
print(f'IF_EXPORT_FEATURES_TARGETS = {IF_EXPORT_FEATURES_TARGETS}')
print(f'EXPORT_DIR = {EXPORT_DIR}')
print(f'EXPORT_FEATURES_TARGETS_FILE_NAME = {EXPORT_FEATURES_TARGETS_FILE_NAME}')
print(50*"=")

USER IO FLAGS:
LOAD_DATA = True
CORE DATA FILE NAME = training_data_complete_20260116_224238.pkl
META DATA FILE NAME = metadata_20260116_224238.json
BASE DIR ALL = ../data/training
IF_EXPORT_FEATURES_TARGETS = True
EXPORT_DIR = ../data/processed/
EXPORT_FEATURES_TARGETS_FILE_NAME = features_targets_training_data_complete_20260116_224238.pkl


## 2. Load training data

Loads from paths above; populates `df_data` and optional `df_meta`.


In [3]:
# 1. Set up file paths and load data (notebook lives under notebooks/; paths relative to that)
# 2. Load the dataset and metadata (or auto-discover latest when LOAD_DATA=False)
df_data = pd.DataFrame()
df_meta = {}

if IF_LOAD_DATA:
    print(50*"=")
    # User specified: load named files (edit names below if needed)
    FILE_PATH_DATA = os.path.join(BASE_DIR_ALL, CORE_DATA_FILE_NAME)
    FILE_PATH_META = os.path.join(BASE_DIR_ALL, META_DATA_FILE_NAME)
    if os.path.isfile(FILE_PATH_DATA):
        df_data = pd.read_pickle(FILE_PATH_DATA)
        n_before = len(df_data)
        df_data = df_data.dropna()
        n_dropped = n_before - len(df_data)
        if n_dropped > 0:
            print(f'Dropped {n_dropped} rows with NaN ({n_before} -> {len(df_data)}).')
        if os.path.isfile(FILE_PATH_META):
            with open(FILE_PATH_META, 'r') as f:
                df_meta = json.load(f)
        print(f'Data loaded :\n{CORE_DATA_FILE_NAME}, \n{META_DATA_FILE_NAME}')
    else:
        print(f'File not found: {FILE_PATH_DATA}')

print(50*"=")

Data loaded :
training_data_complete_20260116_224238.pkl, 
metadata_20260116_224238.json


### 2.1 Data overview

Shape, unique `reactant_type`, and column list. Gated by **PRINT_ON_SCREEN**.


In [4]:
# Data exploration (only when we have data)
if len(df_data) == 0:
    print("No data loaded. Skip exploration or load data in the previous cell.")
elif PRINT_ON_SCREEN:
    print(50*"=")
    print("Dataframe shape:", df_data.shape[0], "rows and", df_data.shape[1], "columns")
    print(50*"")
    if 'reactant_type' in df_data.columns:
        print('Unique reactant types:', df_data['reactant_type'].unique())
    print(50*"")
    print("Dataframe columns:", df_data.columns.to_list())
    print(50*"=")
else:
    print("Data exploration output skipped (PRINT_ON_SCREEN=False).")

Dataframe shape: 152442 rows and 328 columns

Unique reactant types: ['n-Hexane']

Dataframe columns: ['reactant_type', 'initial_temperature_K', 'initial_pressure_Pa', 'reactor_length_m', 'reactor_diameter_m', 'mass_flow_rate_kgps', 'heat_flux_Wm2', 'z_position_m', 'relative_position', 'temperature_K', 'pressure_Pa', 'velocity_ms', 'density_kgm3', 'heat_capacity_cp_JkgK', 'heat_capacity_cv_JkgK', 'mean_molecular_weight_kgkmol', 'enthalpy_Jkg', 'entropy_JkgK', 'internal_energy_Jkg', 'gibbs_free_energy_Jkg', 'viscosity_Pas', 'thermal_conductivity_WmK', 'Y_Water', 'X_Water', 'Y_Ar', 'X_Ar', 'Y_He', 'X_He', 'Y_Ne', 'X_Ne', 'Y_N2', 'X_N2', 'Y_C6H14(1)', 'X_C6H14(1)', 'Y_Benzene(2)', 'X_Benzene(2)', 'Y_C8H10(3)', 'X_C8H10(3)', 'Y_C4H6(4)', 'X_C4H6(4)', 'Y_C4H8(5)', 'X_C4H8(5)', 'Y_Toluene(6)', 'X_Toluene(6)', 'Y_Styrene(7)', 'X_Styrene(7)', 'Y_C2H4(8)', 'X_C2H4(8)', 'Y_C3H6(9)', 'X_C3H6(9)', 'Y_C5H6(10)', 'X_C5H6(10)', 'Y_C3H7(13)', 'X_C3H7(13)', 'Y_C4H9(14)', 'X_C4H9(14)', 'Y_C2H5(15)', 'X_

## 3. Organize columns by category

Splits columns into logical groups (inlet, reactor, operating, spatial, state, thermo, species Y/X). Builds **`df_features`** (model inputs) and **`df_target`** (model outputs) for ML.


In [5]:
# Organize data into logical categories for better analysis and ML
def organize_data_columns(df):
    """Organize dataframe columns into logical categories."""
    all_cols = df.columns.tolist()
    categories = {
        'inlet_conditions': ['initial_temperature_K', 'initial_pressure_Pa'],
        'reactor_design': ['reactor_length_m', 'reactor_diameter_m'],
        'operating_conditions': ['mass_flow_rate_kgps', 'heat_flux_Wm2'],
        'spatial_coordinates': ['z_position_m', 'relative_position'],
        'state_variables': ['temperature_K', 'pressure_Pa', 'velocity_ms', 'density_kgm3'],
        'thermodynamic_properties': [
            'heat_capacity_cp_JkgK', 'heat_capacity_cv_JkgK', 'mean_molecular_weight_kgkmol',
            'enthalpy_Jkg', 'entropy_JkgK', 'internal_energy_Jkg', 'gibbs_free_energy_Jkg',
            'viscosity_Pas', 'thermal_conductivity_WmK'],
        'species_mass_fractions': [c for c in all_cols if c.startswith('Y_')],
        'species_mole_fractions': [c for c in all_cols if c.startswith('X_')]   
    }
    organized = {}
    for category, cols in categories.items():
        existing = [c for c in cols if c in all_cols]
        if existing:
            organized[category] = existing
    return organized

# Run organization only when we have data
data_categories = {}
df_boundary = df_reactor = df_operating = df_spatial = None
df_state = df_thermo = df_species_Y = df_species_X = df_main = None

#########################################################
# SEPARATE DATA INTO CATEGORIES
#########################################################
if len(df_data) > 0:
    data_categories = organize_data_columns(df_data)
    print("=" * 70)
    print("DATA ORGANIZATION AND FEATURES -- TARGETS SETUP")
    print("=" * 70)
    df_boundary = df_data[data_categories['inlet_conditions']] if 'inlet_conditions' in data_categories else None
    df_reactor = df_data[data_categories['reactor_design']] if 'reactor_design' in data_categories else None
    df_operating = df_data[data_categories['operating_conditions']] if 'operating_conditions' in data_categories else None
    df_spatial = df_data[data_categories['spatial_coordinates']] if 'spatial_coordinates' in data_categories else None
    
    df_state = df_data[data_categories['state_variables']] if 'state_variables' in data_categories else None
    df_thermo = df_data[data_categories['thermodynamic_properties']] if 'thermodynamic_properties' in data_categories else None
    df_species_Y = df_data[data_categories['species_mass_fractions']] if 'species_mass_fractions' in data_categories else None
    df_species_X = df_data[data_categories['species_mole_fractions']] if 'species_mole_fractions' in data_categories else None
    #########################################################
    # MAIN INPUT FEATURES
    #########################################################
    # PFR INPUT PARAMETERS SET TO BE USED AS FEATURES 
    # 7 input features + 1 reactant 
    main_input_features = (
        data_categories.get('inlet_conditions', []) +
        data_categories.get('operating_conditions', []) +
        data_categories.get('reactor_design', []) +
        data_categories.get('spatial_coordinates', [])
        )

    main_input_features = [c for c in main_input_features if c in df_data.columns]
    df_features = df_data[main_input_features] if main_input_features else pd.DataFrame()
    print(f'MODEL RAW INPUT FEATURES = {df_features.columns.to_list()}')
    #########################################################
    # MAIN OUTPUT FEATURES
    #########################################################
    # N output features 
    main_output_features = (
        data_categories.get('state_variables', []) +
        data_categories.get('species_mass_fractions', []) +
        data_categories.get('species_mole_fractions', []) +
        data_categories.get('thermodynamic_properties', []))
    main_output_features = [c for c in main_output_features if c in df_data.columns]
    df_target = df_data[main_output_features] if main_output_features else pd.DataFrame()
    print(f'MODEL RAW OUTPUT TARGETS = {df_target.columns.to_list()}')
else:
    print("No data to organize. Load data in the previous cell.")

DATA ORGANIZATION AND FEATURES -- TARGETS SETUP
MODEL RAW INPUT FEATURES = ['initial_temperature_K', 'initial_pressure_Pa', 'mass_flow_rate_kgps', 'heat_flux_Wm2', 'reactor_length_m', 'reactor_diameter_m', 'z_position_m', 'relative_position']
MODEL RAW OUTPUT TARGETS = ['temperature_K', 'pressure_Pa', 'velocity_ms', 'density_kgm3', 'Y_Water', 'Y_Ar', 'Y_He', 'Y_Ne', 'Y_N2', 'Y_C6H14(1)', 'Y_Benzene(2)', 'Y_C8H10(3)', 'Y_C4H6(4)', 'Y_C4H8(5)', 'Y_Toluene(6)', 'Y_Styrene(7)', 'Y_C2H4(8)', 'Y_C3H6(9)', 'Y_C5H6(10)', 'Y_C3H7(13)', 'Y_C4H9(14)', 'Y_C2H5(15)', 'Y_C6H13(16)', 'Y_H(17)', 'Y_C5H11(18)', 'Y_CH3(19)', 'Y_C6H13(20)', 'Y_C6H13(21)', 'Y_C4H9(22)', 'Y_CC(23)', 'Y_C2H3(26)', 'Y_C3H7(28)', 'Y_CCC(29)', 'Y_C5H10(31)', 'Y_C5H11(32)', 'Y_CH4(35)', 'Y_C3H5(36)', 'Y_C4H7(37)', 'Y_C4H7(38)', 'Y_C4H7(39)', 'Y_C4H7(40)', 'Y_C4H8(41)', 'Y_C3H5(43)', 'Y_C3H5(44)', 'Y_C3H6(45)', 'Y_H2(47)', 'Y_C6H12(50)', 'Y_C6H12(62)', 'Y_C4H9(66)', 'Y_C5H9(85)', 'Y_C5H9(86)', 'Y_C5H9(87)', 'Y_C5H9(88)', 'Y_C5H1

## 4. Export features & targets

If **IF_EXPORT_FEATURES_TARGETS** is True, saves `df_features` and `df_target` to **EXPORT_DIR** as a single pickle.

In [6]:
if IF_EXPORT_FEATURES_TARGETS and df_features is not None and df_target is not None:
    os.makedirs(EXPORT_DIR, exist_ok=True)
    export_path = os.path.join(EXPORT_DIR, EXPORT_FEATURES_TARGETS_FILE_NAME)
    with open(export_path, 'wb') as f:
        pickle.dump({'df_features': df_features, 'df_target': df_target}, f)
    print(50*"=")
    print(f'EXPORT FEATURES AND TARGETS TO:\n{export_path}')
    print(50*"=")
elif IF_EXPORT_FEATURES_TARGETS:
    print(50*"=")
    print('EXPORT SKIPPED')
    print(50*"=")
else:
    print(50*"=")
    print('EXPORT DISABLED (IF_EXPORT_FEATURES_TARGETS=False).')
    print(50*"=")

EXPORT FEATURES AND TARGETS TO:
../data/processed/features_targets_training_data_complete_20260116_224238.pkl
